# CrashLens End-to-End Test

This notebook demonstrates CrashLens tracking real GPU workloads with actual CUDA/PyTorch.

**Requirements:**
- GPU runtime (Runtime → Change runtime type → GPU)
- CrashLens backend deployed (Railway, etc.)
- API key (if required by your backend)

## 1. Setup

In [ ]:
# Install CrashLens SDK
!pip install -q requests torch torchvision

# Install crashlens from source (for development)
# !git clone https://github.com/kavinsaravan/ML-Failure-Doctor.git
# !pip install -e ML-Failure-Doctor/crashlens-sdk

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Configure your backend URL and API key
BACKEND_URL = "https://your-backend.railway.app"  # Replace with your backend URL
API_KEY = os.getenv("CRASHLENS_API_KEY", None)  # Optional: set if backend requires auth

# Verify GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Initialize CrashLens Tracker

In [ ]:
# For now, use direct HTTP requests (replace with SDK once published)
import requests
import time
import traceback

class SimpleTracker:
    def __init__(self, backend_url, api_key=None):
        self.backend_url = backend_url.rstrip('/')
        self.headers = {'Content-Type': 'application/json'}
        if api_key:
            self.headers['Authorization'] = f'Bearer {api_key}'
    
    def create_workload(self, name):
        resp = requests.post(
            f"{self.backend_url}/workloads",
            json={"name": name, "type": "ML_JOB", "status": "running"},
            headers=self.headers
        )
        resp.raise_for_status()
        return resp.json()["id"]
    
    def update_workload(self, wid, status, logs="", exit_code=0):
        requests.put(
            f"{self.backend_url}/workloads/{wid}",
            json={"status": status, "job_logs": logs, "exit_code": exit_code},
            headers=self.headers
        )
    
    def diagnose(self, wid):
        resp = requests.post(
            f"{self.backend_url}/workloads/{wid}/diagnose",
            headers=self.headers
        )
        resp.raise_for_status()
        return resp.json()

tracker = SimpleTracker(BACKEND_URL, API_KEY)
print("✅ Tracker initialized")

## 3. Test 1: Successful Training

In [ ]:
# Simple CNN model
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 13 * 13, 10)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(-1, 16 * 13 * 13)
        return self.fc1(x)

# Track the training
workload_id = tracker.create_workload("E2E Test - Successful Training")
logs = []

try:
    # Create dummy data
    X = torch.randn(100, 3, 28, 28)
    y = torch.randint(0, 10, (100,))
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=32)
    
    # Train
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())
    
    logs.append(f"Training on {device}")
    
    for epoch in range(3):
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        logs.append(f"Epoch {epoch+1}/3 - Loss: {loss.item():.4f}")
        print(logs[-1])
    
    # Success
    tracker.update_workload(workload_id, "succeeded", "\n".join(logs), 0)
    print(f"✅ Workload {workload_id} succeeded")
    
except Exception as e:
    error_log = "\n".join(logs) + "\n" + traceback.format_exc()
    tracker.update_workload(workload_id, "failed", error_log, 1)
    print(f"❌ Workload {workload_id} failed")
    raise

## 4. Test 2: GPU Out of Memory (Intentional Failure)

In [ ]:
# Create a model that's too large for GPU
workload_id = tracker.create_workload("E2E Test - GPU OOM")
logs = []

try:
    # Only run on CUDA, skip on CPU
    if not torch.cuda.is_available():
        logs.append("Skipping OOM test - no GPU available")
        tracker.update_workload(workload_id, "succeeded", "\n".join(logs), 0)
        print("⏭️  Skipped OOM test (no GPU)")
    else:
        # Intentionally allocate too much memory
        logs.append("Attempting to allocate large tensor...")
        print(logs[-1])

        # Allocate ~40GB (exceeds T4's 15GB)
        huge_tensor = torch.randn(100000, 100000, device="cuda")

        logs.append("Allocation succeeded (unexpected!)")
        tracker.update_workload(workload_id, "succeeded", "\n".join(logs), 0)
    
except RuntimeError as e:
    error_log = "\n".join(logs) + "\n" + traceback.format_exc()
    tracker.update_workload(workload_id, "failed", error_log, 1)
    print(f"❌ Workload {workload_id} failed (expected!)")
    print(f"Error: {str(e)}")
    
    # Get AI diagnosis
    print("\n🔍 Running AI diagnosis...")
    diagnosis = tracker.diagnose(workload_id)
    print(f"Failure Type: {diagnosis['failure_type']}")
    print(f"Root Cause: {diagnosis['root_cause']}")
    print(f"\nRecommended Fix:\n{diagnosis['recommended_fix']}")

## 5. Test 3: Dependency Error

In [ ]:
workload_id = tracker.create_workload("E2E Test - Dependency Error")
logs = []

try:
    # Try to import a non-existent package
    logs.append("Importing dependencies...")
    import nonexistent_package_12345
    
    tracker.update_workload(workload_id, "succeeded", "\n".join(logs), 0)
    
except ModuleNotFoundError as e:
    error_log = "\n".join(logs) + "\n" + traceback.format_exc()
    tracker.update_workload(workload_id, "failed", error_log, 1)
    print(f"❌ Workload {workload_id} failed (expected!)")
    
    # Get AI diagnosis
    print("\n🔍 Running AI diagnosis...")
    diagnosis = tracker.diagnose(workload_id)
    print(f"Failure Type: {diagnosis['failure_type']}")
    print(f"Confidence: {diagnosis['confidence']:.2%}")
    print(f"Root Cause: {diagnosis['root_cause']}")

## 6. Verify Results in Dashboard

Visit your CrashLens dashboard to see:
- All 3 workloads tracked
- Real GPU metrics from CUDA
- AI diagnosis for failed workloads
- Wasted GPU time calculations

**Dashboard URL:** Replace with your deployed frontend URL

In [ ]:
# List all workloads
resp = requests.get(f"{BACKEND_URL}/workloads")
workloads = resp.json()

print("\n📊 All Workloads:")
for w in workloads[-5:]:  # Show last 5
    status_emoji = "✅" if w["status"] == "succeeded" else "❌"
    print(f"{status_emoji} [{w['id']}] {w['name']} - {w['status']}")

print(f"\n🔗 View dashboard: {BACKEND_URL.replace(':8080', ':3000')}")